In [ ]:
# Building chatbot with multiple  tools using langgraph 
# Aim --> Create a chatbot with tool capabilities from arxiv,wikipedia search and some functions 
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper


In [ ]:
import arxiv

# Create ArxivQueryRun tool
arxiv_wrapper = ArxivAPIWrapper()
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

# 1. Initialize the official native client
client = arxiv.Client()

# 2. Configure your search query
search = arxiv.Search(
    query="quantum computing",
    max_results=2
)

# 3. Fetch results using the client
results = client.results(search)

# 4. Print out your document details
for result in results:
    print(f"Title: {result.title}")
    print(f"Summary: {result.summary[:]}\n")

In [ ]:
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results = 2,doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

In [ ]:
try:
    result = wiki.invoke({"query":"Brian Lara"})
    print(result)
except Exception as e:
    print(f"Error occurred: {type(e).__name__}: {str(e)}")
    print("Wikipedia API may be temporarily unavailable. Try again later.")

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os 
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
tavily = TavilySearchResults()
tavily.invoke("How much percent rain deficiency is there in jaipur? ")


In [ ]:
# combine all tools in the list 
tools = [arxiv_tool, wiki, tavily]

In [ ]:
# initialize my llm model 
from langchain_groq import ChatGroq
llm = ChatGroq(model = "openai/GPT-OSS-20B")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langchain_core.messages import HumanMessage,AIMessage
res = llm_with_tools.invoke([HumanMessage(content = "What is recent AI News")])
res.pretty_print()

In [ ]:
## State schema 
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from typing import Annotated
from langgraph.graph.message import add_messages
class State(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]
    



In [ ]:
# Entire chatbot with langgraph 
from IPython.display import Image,display 
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode,tools_condition

## Node definition 
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

# Create tool node
tool_node = ToolNode(tools)

builder = StateGraph(State)
builder.add_node("tool_calling_llm",tool_calling_llm)
builder.add_node("tools", tool_node)

# Add edges
builder.add_edge(START,"tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition, {"tools": "tools", "__end__": END})
builder.add_edge("tools","tool_calling_llm")

# Compile the graph
graph = builder.compile()

In [ ]:
result = graph.invoke({"messages":[HumanMessage(content = "What is 5 plus 8")]})
for m in result["messages"]:
    m.pretty_print()

In [ ]:
## Memory saver
## Langgraph can use a checkpointer to save the graph state after each 
## step. 
## One of the easiest checkpoint to use is memorysaver,an in-memory
#  key-value store for graph state.
# Specify the thread
messages = [HumanMessage(content= "Divide that by five")]
messages = graph.invoke({"messages":messages})
for m in messages["messages"]:
    m.pretty_print()


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
graph_memory = builder.compile(checkpointer = memory)

# View 
display(Image(graph_memory.get_graph().draw_mermaid_png()))


In [ ]:
# Specify the thread
config = {"configurable":{"thread_id":"1"}}
messages = [HumanMessage(content = "Add 20 and 55 ")]
messages = graph_memory.invoke({"messages":messages},config = config)
for m in messages["messages"]:
    m.pretty_print()
messages = [HumanMessage(content = "Divide that by 5")]
messages = graph_memory.invoke({"messages":messages},config = config)
for m in messages["messages"]:
    m.pretty_print()


In [ ]:
## Streaming 
## Methods: .stream() and .astream()
## These methods are sync and async methods for streaming back results.
## Additional parameters in streaming modes for graph state: 
## 1) values: this streams the full state of the graph after each node is called
## 2) update: this streams updates to the state of the graph after each node is called


In [ ]:
# Stream responses using stream method 
config = {"configurable":{"thread_id":"2"}}
for chunk in graph_memory.stream({"messages":"Hi my name is Divyanshu Jain and I Like cricket"},config = config,stream_mode ="updates")

